# 🧊 Level 4: 3D Visualization & Regression

**[📖 Want a detailed explanation? Read the Manual (Streamlit App)](https://bookseal-seoul-apt-price-prediction.streamlit.app/Level_4_3D_Regression)**

So far we used **Area** and **District**.
But what about **Building Year**? Newer apartments are more expensive!

- **Goal**: Predict price using **Area** + **District** + **Building Year**.
- **Challenge**: How to visualize 3 numeric variables? We need 3D plots!

### 1. Load Data
We load the data and ensure we have the 'year' column.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Import for 3D plotting
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

url = "https://github.com/bookseal/seoul-apt-price-prediction/raw/main/data/sample.parquet"
df = pd.read_parquet(url)

# Create synthetic 'year' if missing (for the sake of demo)
np.random.seed(42)
if 'year' not in df.columns:
    df['year'] = np.random.randint(1985, 2024, len(df))
    print("Generated synthetic 'year' data.")

print(f"Data Loaded! Rows: {len(df):,}")
df.head()

### 2. Prepare Training Data
We now have 2 numeric features (**Area**, **Year**) and 1 categorical feature (**District**).

In [ ]:
# 1. One-Hot Encode District
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_district = encoder.fit_transform(df[['district']])

# 2. Get Numeric Features
X_numeric = df[['area_m2', 'year']].values

# 3. Combine
X = np.hstack([X_numeric, X_district])
y = df['price_10k_krw'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Features: Area + Year + {X_district.shape[1]} Districts")

### 3. Visualizing in 3D (Area x Year x Price)
Before training, let's explore the data in 3D space.
Note: 3D plots are static on GitHub/NBViewer but rotatable in local Jupyter.

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Sample data for faster plotting
sample = df.sample(n=1000, random_state=42)

scatter = ax.scatter(sample['area_m2'], sample['year'], sample['price_10k_krw'], 
                     c=sample['price_10k_krw'], cmap='coolwarm', alpha=0.5)

ax.set_xlabel('Area (m2)')
ax.set_ylabel('Building Year')
ax.set_zlabel('Price (10k KRW)')
ax.set_title('3D View: Area & Year vs Price')

plt.colorbar(scatter, label='Price')
plt.show()

### 4. Train Model
Model learns: $Price = w_1(Area) + w_2(Year) + w_{dist}(District) + b$

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

w_area = model.coef_[0]
w_year = model.coef_[1]

print("Model Learned:")
print(f"Area Effect: +{w_area:.0f} KRW per m²")
print(f"Year Effect: +{w_year:.0f} KRW per year newer")

### 5. Predict
Let's see how much "Newer" helps the price.

In [ ]:
area = 84
district = '강남구'
dist_vec = encoder.transform([[district]])

# Compare 1990 vs 2020
X_old = np.hstack([[area, 1990], dist_vec[0]]).reshape(1, -1)
X_new = np.hstack([[area, 2020], dist_vec[0]]).reshape(1, -1)

price_old = model.predict(X_old)[0]
price_new = model.predict(X_new)[0]

print(f"1990 Apartment Price: {price_old:,.0f}")
print(f"2020 Apartment Price: {price_new:,.0f}")
print(f"New Building Premium: {price_new - price_old:,.0f} (Approx {(price_new - price_old)/10000:.1f} 억원)")